## Notebook to Run Your Own Networks
The following section runs all of the set up. This is followed by some starter code where you can fill in the values for your verification.

In [ ]:
# Run the setup
%run startup.py

# Import classes used to help easily run tests using pybatfish
from verificationPythonSupport import LocationPropertyPair,VerificationQuery,Clause

# Help functions to run tests and examples easier
def create(networkName:str,snapshotName:str,snapshotPath:str):
    '''Creates a batfish session for the snapshot located at `snapshotPath` and is given the provided `networkName` and `snapshotName`.'''
    bf = Session(host="localhost")
    bf.set_network(networkName)
    bf.init_snapshot(snapshotPath, name=snapshotName, overwrite=True)
    return bf

# This function is what includes the call to the pybatfish verification question
def runVerificationQuestion(bf,show_all:bool,query:VerificationQuery):
    '''Returns the result of making the provided VerificationQuery `query` using the provided batfish session `bf`. The provided
    `show_all` flag indicates if all locations results should be displayed or just the most relevant ones.'''
    if query == None:
        return bf.q.safety().answer().frame()
    formatted = query.format()
    if formatted["assumption_locations"] == "":
        result = bf.q.safety(
            target=formatted["target"],
            location=formatted["location"],
            show_all=show_all,
            refine=query.refines()).answer()
    else:
        result = bf.q.safety(
            target=formatted["target"],
            location=formatted["location"],
            assumption_locations=formatted["assumption_locations"],
            assumptions=formatted["assumptions"],
            show_all=show_all,
            refine=query.refines()).answer()
    return result.frame()

def runAndDisplay(networkName:str,snapshotName:str,snapshotPath:str,query:VerificationQuery,show_all=False):
    '''Runs and displays the result of running the VerificationQuery `query` on the snapshot located at `snapshotPath`. The optional `show_all`
    flag is passed to the query with a default value of False.'''
    bf = create(networkName,snapshotName,snapshotPath)
    verificationResult = runVerificationQuestion(bf,show_all,query)
    show(verificationResult)


In [ ]:
# NOTE these are random values used to show different usages

networkName = "your_network_name"
snapshotName = "your_snapshot_name"
snapshotPath = "patht/to/your/snapshots"

target = LocationPropertyPair(location="100.0.0.1 -> 10.0.0.1",property=[Clause(communities=["100:1","100:2"],prefixes=["25.13.0.0/16"])])

assumption1 = LocationPropertyPair(location="100.0.0.4 -> 10.0.0.4",property=[Clause(communities=["!100:3"],prefixes=[]),Clause(communities=[],prefixes=["!25.13.0.0/16"])])
assumption2 = LocationPropertyPair(location="100.0.0.3 -> 10.0.0.3",property=[Clause(communities=["!100:4"],prefixes=[])])

# NOTE remember the default for both show_all and refine are False
query = VerificationQuery(target=target,assumptions=[assumption1,assumption2],refine=True)
runAndDisplay(networkName,snapshotName,snapshotPath,query,show_all=True)